# Exercises XP: RAG with LangChain (Student)

## 0) Configuration et Importations

Dans cette section, nous importons les outils nécessaires :
*   **`datasets`** : Pour charger facilement des données textuelles.
*   **`transformers`** : La bibliothèque de Hugging Face pour manipuler des modèles de langage (LLM).
*   **`langchain`** : Un framework qui permet de 'chaîner' différents composants (recherche + IA) pour créer une application.
*   **`FAISS`** : Une base de données optimisée pour stocker des vecteurs et faire de la recherche ultra-rapide.

In [1]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
from typing import List

# Datasets: permet de télécharger des jeux de données open-source
from datasets import load_dataset
# Pipeline: interface simplifiée de Hugging Face pour utiliser un modèle
from transformers import pipeline

# LangChain Core: les objets de base comme 'Document' (texte + métadonnées)
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# Text Splitters: outils pour découper les longs textes en petits morceaux (chunks)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings: convertit du texte en listes de nombres (vecteurs)
from langchain_community.embeddings import HuggingFaceEmbeddings
# Vector Store (FAISS): notre index de recherche pour retrouver les documents pertinents
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# LangChain Integration: connecte les modèles Hugging Face à la logique LangChain
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

## 1) Load dataset and convert to Documents


In [15]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    # On crée un objet Document pour chaque ligne du dataset
    # page_content contient le texte principal
    # metadata contient des informations additionnelles comme la source (URL/nom du fichier)
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

Documents: 200
Example: {'source': 'gradio-app/gradio/blob/main/demo/blocks_random_slider/run.ipynb'}
 Gradio Demo: blocks_random_slider


```
!pip install -q gradio 
```


```

import gradio as gr


def func(slider_1, slider_2):
    return slider_1 * 5 + slider_2


with gr.Blocks() as demo:
    slider = gr.Slider(minimum=-10.2, maximum=15, label="Random Slider (Static)", randomize=True)
    slider_1 = gr.Slider(minimum=100, maximum=200, label="Ran


## 2) Split into chunks


In [4]:
# Le chunking est crucial : si le texte est trop long, il ne tiendra pas dans la 'mémoire' du LLM
chunk_size = 500   # Nombre de caractères par morceau
chunk_overlap = 50 # Chevauchement pour ne pas couper une phrase importante entre deux morceaux

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""] # Définit où couper par priorité
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

Chunks: 5966
First chunk: {'source': 'huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx'}
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 




## 3) Vector store + retriever (FAISS)


In [5]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

# On transforme le texte en vecteurs numériques (embeddings)
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

# FAISS indexe ces vecteurs pour permettre une recherche de similarité rapide
vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE # Mesure l'angle entre les vecteurs
)

# Le retriever est l'interface qui va chercher les 4 documents les plus proches de la question
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

/tmp/ipykernel_1320/4125184527.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever ready


## 4) Build the RAG chain


In [9]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

# Nous utilisons un modèle léger (FLAN-T5) pour l'exercice
llm_id = "google/flan-t5-small"

# 1. Création du pipeline de génération
# Le pipeline transforme l'entrée texte en une sortie textuelle intelligible
hf = pipeline(
    task="text-generation",
    model=llm_id,
    model_kwargs={"device_map": "auto"},
    max_new_tokens=100
)

# 2. On encapsule ce pipeline dans un objet LangChain pour l'utiliser dans nos chaînes
llm = HuggingFacePipeline(pipeline=hf)

# 3. On crée la chaîne 'RetrievalQA'
# Cette chaîne orchestre tout : elle prend la question, interroge la base FAISS (retriever),
# puis envoie la question + les documents trouvés au LLM (llm).
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff", # 'stuff' combine tous les documents trouvés dans un seul prompt
    return_source_documents=True
)

print("La chaîne RAG est maintenant corrigée et prête !")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

La chaîne RAG est maintenant corrigée et prête !


## 5) Démonstration : Comparaison RAG vs No-RAG

Cette étape finale est la plus importante pour comprendre l'intérêt du RAG :
*   **Sans RAG** : Le modèle répond en utilisant uniquement ses connaissances apprises lors de son entraînement initial. Il peut se tromper ou ne pas connaître les documents spécifiques de Hugging Face.
*   **Avec RAG** : Nous fournissons au modèle les morceaux de texte pertinents trouvés dans notre base FAISS. Il s'en sert comme 'antisèche' pour donner une réponse précise et sourcée.

In [10]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# --- Approche 1 : Sans RAG ---
# On pose la question directement au modèle sans contexte supplémentaire.
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# --- Approche 2 : Avec RAG ---
# La chaîne 'qa' va d'abord chercher les documents, puis répondre.
rag_result = qa.invoke({"query": q})

print("Question posée :", q)
print("\n--- Réponse SANS RAG (Connaissances internes) ---\n", no_rag_answer)
print("\n--- Réponse AVEC RAG (Basée sur les docs) ---\n", rag_result["result"])

print("\n--- Sources utilisées ---")
# Il est essentiel d'afficher les sources pour vérifier la fiabilité de l'IA.
for i, doc in enumerate(rag_result["source_documents"]):
    print(f"Source {i+1}: {doc.metadata.get('source')}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question posée : How can I retrieve a model from the Hugging Face Hub?

--- Réponse SANS RAG (Connaissances internes) ---
 Answer the question. If you are not sure, say you are not sure.

Question: How can I retrieve a model from the Hugging Face Hub?
Answer:

--- Réponse AVEC RAG (Basée sur les docs) ---
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

## Joining Hugging Face and installation

To share models in the Hub, you will need to have a user. Create it on the [Hugging Face website](https://huggingface.co/join).

Now when you navigate to your Hugging Face profile, you should see your newly created model repository. Clicking on the **Files** tab will display all the files you've uploaded to the repository.

For more details on how to create and upload files to a repository, refer to the Hub documentation [here](https://huggingface.co/docs/hub/how-to-upstream).

## 